# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 |
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
from google.colab import files

uploaded = files.upload()

In [2]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 23.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.3 MB/s eta 0:00:00
Setup complete!


In [3]:
import os, re, json, time, random, warnings
print(os.listdir())
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

['.config', 'gold_standard_100.csv', 'llm_labels_150.csv', '.env', 'text_embedder.pt', 'requirements.txt', 'weak_labels_200.csv', 'evaluator.py', 'sample_data']
Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [4]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs"
    )
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label

prob=clf.predict_proba(vec.transform(llm['review']))
pred=clf.predict(vec.transform(llm['review']))
confidence=prob.max(axis=1)
llm['prediction']=pred
llm['confidence']=confidence
llm_filter=llm[(llm['confidence']>=0.65) & (llm['label']==llm['prediction'])]
llm_filter_final=llm_filter.drop(columns=['prediction','confidence'])
print(llm_filter_final)

#  1d. Merge & deduplicate
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
consolidated_base=pd.concat([gold,weak,llm_filter_final],ignore_index=True)
consolidated_base=consolidated_base.drop_duplicates(subset=['review'])
consolidated_base.to_csv("consolidated_base.csv",index=False)
print("Final consolidated size:", len(consolidated_base))

#  1e. Class distribution analysis
# TODO: Count per class, identify minority, plot and save class_distribution.png
counts=consolidated_base['label'].value_counts()
minority_class=counts.idxmin()

print(counts)
print("Minority class:", minority_class)

plt.figure(figsize=(6,4))
counts.plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class Label")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.show()

Gold: 100, Weak: 220, LLM: 150
                                                review     label
0    Ideally, this should have been great, but some...   Neutral
9    This movie is a triumph in every sense. Don't ...  Positive
20   Finally, a movie that respects the audience. I...  Positive
26   In my opinion, i struggled to sit through the ...  Negative
35   The visual effects were delightful and added s...  Positive
47   It perfectly balances humor and drama. Wow, si...  Positive
48   It felt like a rough draft that was never edit...  Negative
50   I struggled to sit through the first half. Not...  Negative
52   The score was frankly convoluted. I tried to l...  Negative
55   It serves its purpose as popcorn filler. Visua...   Neutral
59   I was completely blown away by this film. It m...  Positive
66   An absolute train wreck of a movie. A complete...  Negative
67   To be fair, it’s a weird mix of brilliance and...   Neutral
69   I have never been so bored in my life. Zero st...  Neg

In [5]:
#  1f. Augmentation functions
def get_wordnet_pos(tag):
    """Map POS tag to WordNet format"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    # TODO: Implement using nltk.corpus.wordnet, nltk.pos_tag, word_tokenize
    words = word_tokenize(text)
    tagged = pos_tag(words)
    n_replace = max(1, int(len(words) * replace_frac))
    # candidate indices (skip sentiment words)
    candidates = [i for i, (w, _) in enumerate(tagged) if w.lower() not in PRESERVE_WORDS]
    random.shuffle(candidates)
    replace_indices = candidates[:n_replace]
    new_words = words.copy()
    for i in replace_indices:
        word, tag = tagged[i]
        wn_tag = get_wordnet_pos(tag)
        if wn_tag is None:
            continue
        synsets = wordnet.synsets(word, pos=wn_tag)
        if synsets:
            lemmas = synsets[0].lemma_names()
            if lemmas:
                synonym = lemmas[0].replace("_", " ")
                if synonym.lower() != word.lower():
                    new_words[i] = synonym
    return " ".join(new_words)

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    # TODO: Implement with error handling and rate-limit sleep
    try:
        hindi = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.5)
        back = GoogleTranslator(source=mid, target=src).translate(hindi)
        time.sleep(0.5)
        return back
    except Exception as e:
        return text  # fallback if error

def jaccard_similarity(a, b):
    set_a = set(a.lower().split())
    set_b = set(b.lower().split())
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    if union == 0:
        return 0
    return intersection / union

def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    score = jaccard_similarity(original, augmented)
    return 0.30 <= score <= 0.95

#  1g. Apply augmentation to minority class
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
# TODO: Save augmented_classical.csv
augmented_data = []
minority_df = consolidated_base[consolidated_base['label'] == minority_class]
for _, row in minority_df.iterrows():
    text = row['review']
    label = row['label']

    # Synonym replacement
    aug1 = synonym_replacement(text)
    if quality_filter(text, aug1):
        augmented_data.append({"review": aug1, "label": label})

    #Back translation
    aug2 = back_translate(text)
    if quality_filter(text, aug2):
        augmented_data.append({"review": aug2, "label": label})
augmented_classical = pd.DataFrame(augmented_data)
augmented_classical = augmented_classical.drop_duplicates(subset=['review'])
augmented_classical.to_csv("augmented_classical.csv", index=False)
print("Augmented samples:", len(augmented_classical))

Augmented samples: 121


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [6]:
gold.head()
weak.head()

,review,label
0,It’s a weird mix of brilliance and stupidity. ...,Neutral
1,I struggled to sit through the first half. Not...,Negative
2,I struggled to sit through the first half. The...,Negative
3,I'm honestly still trying to process what I ju...,Negative
4,A refreshing take on a tired genre. It perfect...,Negative


In [7]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "LLM Review Generation"
    }
)

print("Client ready")

#  2a. Design your few-shot prompt
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
p=gold[gold['label']=='Positive']
n=gold[gold['label']=='Negative']
neutral=gold[gold['label']=='Neutral']

pos=p.sample(n=1,random_state=42)
neg=n.sample(n=1,random_state=42)
neu=neutral.sample(n=1,random_state=42)
opt=gold.sample(n=1,random_state=42) #optional

sample_data=pd.concat([pos,neg,neu,opt],ignore_index=True)
print(sample_data)

example_block=""
for i,row in enumerate(sample_data.itertuples()):
  example_block += f"""
  {{
    "review": "{row.review}",
    "sentiment": "{row.label}",
    "movie": "Example Movie"
  }}
  """

PROMPT_TEMPLATE = f"""You are a movie review generator. ...
your task is to generate movie reviews with sentiment labels
below are examples:

{example_block}

Now generate 20 new movie reviews.

IMPORTANT DISTRIBUTION:
- 10 Positive
- 7 Negative
- 3 Neutral

Instructions:
1.Each review should be 1-3 sentences long.
2.label must be one of the Positive, Negative, Neutral
3.Return the output only in Json fromat for example:
[
  {{"review": "text here", "sentiment": "Positive", "movie": "movie name"}},
  {{"review": "text here", "sentiment": "Negative", "movie": "movie name"}}
]

STRICT FORMAT RULE:
- Output MUST be valid JSON
- Do NOT include ``` or markdown
- Do NOT include text before or after JSON

DO NOT include any explanation or text outside JSON.
ONLY return valid JSON array.

"""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)


#  2b. Generate reviews in batches
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors
all_reviews = []

for i in range(0, 300, 20):
    success = False   # added

    for attempt in range(2):   # added retry
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": PROMPT_TEMPLATE}],
                temperature=0.9
            )

            output_text = response.choices[0].message.content
            json_match = re.search(r'\[.*\]', output_text, re.DOTALL)

            if json_match:
                try:
                    batch_data = json.loads(json_match.group())

                    if len(batch_data) != 20:   # added size check
                        print(f"Batch {i} wrong size: {len(batch_data)} (attempt {attempt+1})")
                        continue
                    valid_batch = True
                    for item in batch_data:
                        if not all(k in item for k in ["review", "sentiment", "movie"]):
                            valid_batch = False
                        if item.get("sentiment") not in ["Positive", "Negative", "Neutral"]:
                            valid_batch = False
                    if not valid_batch:
                        print(f"Batch {i} invalid structure (attempt {attempt+1})")
                        continue
                    all_reviews.extend(batch_data)
                    print(f"Batch {i} success: {len(batch_data)} reviews")
                    success = True
                    break
                except json.JSONDecodeError:
                    print(f"Batch {i} JSON parsing error (attempt {attempt+1})")
            else:
                print(f"Batch {i} JSON not found (attempt {attempt+1})")

        except Exception as e:
            print(f"Batch {i} error (attempt {attempt+1}): {e}")

        time.sleep(2)

    if not success:
        print(f"Batch {i} FAILED after retries")

    time.sleep(3)


llm_df = pd.DataFrame(all_reviews)
print(f"Total reviews generated: {len(llm_df)}")
print(llm_df['sentiment'].value_counts())
#  2c. Diversity metrics
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score
def compute_self_bleu(reviews):   # renamed
  smoothie=SmoothingFunction().method1
  scores=[]

  tokenize_reviews=[word_tokenize(review.lower()) for review in reviews]

  for i in range(len(tokenize_reviews)):

    candidate=tokenize_reviews[i]

    references=tokenize_reviews[:i]+tokenize_reviews[i+1:]

    if(len(references)==0):
      continue
    score=sentence_bleu(references,candidate,smoothing_function=smoothie)

    scores.append(score)

  return scores

diversity_results={}

for sentiment in ['Positive','Negative','Neutral']:

  subset=llm_df[llm_df['sentiment']==sentiment]['review'].tolist()

  if(len(subset)>1):
    scores=compute_self_bleu(subset)
  else:
    scores=None

  diversity_results[sentiment]=scores

#  2d. Sentiment consistency check
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv

if not llm_df.empty:   # added safety

  probab_llm=clf.predict_proba(vec.transform(llm_df['review']))
  predict_llm=clf.predict(vec.transform(llm_df['review']))
  confidence_llm=probab_llm.max(axis=1)

  llm_df['confidence']=confidence_llm
  llm_df['predicted']=predict_llm

  llm_df['mismatch']=llm_df['predicted']!=llm_df['sentiment']

  flagged = llm_df[llm_df['mismatch']]
  flagged.to_csv("llm_generated_flagged.csv", index=False)


#  2e. Save outputs
# TODO: Save llm_generated_300.csv and diversity_report.txt

llm_df.to_csv("llm_generated_300.csv", index=False)
print("Generated reviews saved to llm_generated_300.csv")

with open("diversity_report.txt", "w") as f:
    for k, v in diversity_results.items():
        if v is not None and len(v) > 0:
            avg_score = sum(v) / len(v)
            status = "PASS" if avg_score < 0.7 else "FAIL"
            f.write(f"{k}: {avg_score:.4f} ({status})\n")
        else:
            f.write(f"{k}: Not enough data\n")

Client ready
                                              review     label
0  A refreshing take on a tired genre. Don't miss...  Positive
1  I have never been so bored in my life. Utterly...  Negative
2  It doesn't leave a lasting impression. It serv...   Neutral
3  I struggled to sit through the first half. A c...  Negative
Batch 0 success: 20 reviews
Batch 20 success: 20 reviews
Batch 40 success: 20 reviews
Batch 60 success: 20 reviews
Batch 80 success: 20 reviews
Batch 100 success: 20 reviews
Batch 120 success: 20 reviews
Batch 140 success: 20 reviews
Batch 160 success: 20 reviews
Batch 180 success: 20 reviews
Batch 200 success: 20 reviews
Batch 220 success: 20 reviews
Batch 240 success: 20 reviews
Batch 260 success: 20 reviews
Batch 280 success: 20 reviews
Total reviews generated: 300
sentiment
Positive    147
Negative    107
Neutral      46
Name: count, dtype: int64
Generated reviews saved to llm_generated_300.csv


In [8]:
llm_df['sentiment'].value_counts()

,count
sentiment,
Positive,147
Negative,107
Neutral,46


In [9]:
accuracy=(llm_df['mismatch']==False).mean()
print(f"consistency accuracy: {accuracy:.4f}")

consistency accuracy: 0.5133


## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [10]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)
def strategic_sampling(df):
  df = df.copy()
  df['len']=df['review'].apply(len)
  pos=df[df['label']=='Positive'].sort_values('len').head(40)
  neg=df[df['label']=='Negative'].sort_values('len').head(40)
  neu=df[df['label']=='Neutral'].sort_values('len').head(20)
  return pd.concat([pos,neg,neu]).drop(columns=['len'])
sample_df=strategic_sampling(consolidated_base)
print(len(sample_df))
print("\nSample distribution:")
print(sample_df['label'].value_counts())
#  3b. Translation pipeline
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

en_to_hi=GoogleTranslator(source='en',target='hi')
hi_to_en = GoogleTranslator(source='hi', target='en')
smoothie = SmoothingFunction().method1
results = []
for _, row in sample_df.iterrows():
    orig_text = row['review']
    label = row['label']
    try:
        # Translate to Hindi
        hindi_text = en_to_hi.translate(orig_text)
        time.sleep(0.5) # Rate limiting

#  3c. Back-translation & BLEU
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"

        # Back-translate to English
        back_translated = hi_to_en.translate(hindi_text)
        time.sleep(0.5)

        # Compute BLEU Score
        ref_tokens = word_tokenize(orig_text.lower()) if orig_text else []
        cand_tokens = word_tokenize(back_translated.lower()) if back_translated else []
        if len(ref_tokens) == 0 or len(cand_tokens) == 0:
            bleu = 0
        else:
            bleu = sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)
        quality_flag = "PASS" if bleu >= 0.3 else "FAIL"

        results.append({
            "review": orig_text,
            "label": label,
            "hindi": hindi_text,
            "back_translated": back_translated,
            "bleu_score": bleu,
            "quality_flag": quality_flag
        })
    except Exception as e:
        print(f"Error translating: {e}")

bilingual_df = pd.DataFrame(results)
#  3d. Sentiment preservation check
# TODO: Predict sentiment on back-translated text, compare with original label

if not bilingual_df.empty:
    pred_back = vec.transform(bilingual_df['back_translated'])
    bilingual_df['predicted_label'] = clf.predict(pred_back)
    bilingual_df['sentiment_preserved'] = bilingual_df['predicted_label'] == bilingual_df['label']
else:
    print("Warning: No translations available")
bilingual_df['final_quality'] = (
    (bilingual_df['bleu_score'] >= 0.3) &
    (bilingual_df['sentiment_preserved'])
)
bilingual_df['final_quality'] = bilingual_df['final_quality'].apply(lambda x: "PASS" if x else "FAIL")
#  3e. Manual verification
# TODO: Print 5 random samples for inspection
print("\n--- Manual Verification (5 Random Samples) ---")
print(bilingual_df[['review', 'hindi', 'back_translated', 'bleu_score', 'quality_flag','final_quality']].sample(5, random_state=42))

#  3f. Save
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag
bilingual_df.to_csv("bilingual_reviews.csv", index=False)
print("\nTask 3 Complete: bilingual_reviews.csv saved.")
print("\nObservation: Translations generally preserve sentiment with minor wording variations.")
print("BLEU PASS rate:", (bilingual_df['bleu_score'] >= 0.3).mean())
print("Sentiment preserved rate:", bilingual_df['sentiment_preserved'].mean())
print("Final PASS rate:", (bilingual_df['final_quality'] == "PASS").mean())


100

Sample distribution:
label
Positive    40
Negative    40
Neutral     20
Name: count, dtype: int64

--- Manual Verification (5 Random Samples) ---
                                               review  \
83  It doesn't leave a lasting impression. It was....   
53  Total garbage. Save your money and watch somet...   
70  The screenplay was frankly pretentious. A frus...   
45  I want my two hours back. It’s a hard pass fro...   
44  A frustrating experience. Avoid this at all co...   

                                                hindi  \
83    यह कोई स्थायी प्रभाव नहीं छोड़ता. यह... ठीक था.   
53         कुल कचरा. अपना पैसा बचाएं और कुछ और देखें।   
70  पटकथा स्पष्ट रूप से दिखावटी थी। एक निराशाजनक अ...   
45  मुझे अपने दो घंटे वापस चाहिए। यह मेरे लिए एक क...   
44          एक निराशाजनक अनुभव. इससे हर कीमत पर बचें.   

                                      back_translated  bleu_score  \
83  It does not leave any lasting effect. It was.....    0.254509   
53  Total garbage. Save y

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [11]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews
def sample_mixed(df, label, n=10):
    subset = df[df['label'] == label].copy()
    subset['len'] = subset['review'].apply(len)

    short = subset.sort_values('len').head(n//2)
    long = subset.sort_values('len', ascending=False).head(n - n//2)

    return pd.concat([short, long]).drop_duplicates().head(n)

pos = sample_mixed(consolidated_base, 'Positive', 10)
neg = sample_mixed(consolidated_base, 'Negative', 10)
neu = sample_mixed(consolidated_base, 'Neutral', 10)

audio_df = pd.concat([pos, neg, neu]).drop(columns=['len']).reset_index(drop=True)

print(len(audio_df))

#  4b. TTS generation
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile
audio_paths = []

for i, row in audio_df.iterrows():
    text = row['review']
    mp3_path = f"audio_samples/sample_{i}.mp3"
    wav_path = f"audio_samples/sample_{i}.wav"

    try:
        # Generate MP3 using gTTS
        tts = gTTS(text=text, lang='en', tld="com")
        tts.save(mp3_path)

        # Load MP3 and convert to WAV
        y, sr = librosa.load(mp3_path, sr=None)
        sf.write(wav_path, y, sr)
        os.remove(mp3_path)
        # Store WAV path
        audio_paths.append(wav_path)

    except Exception as e:
        print(f"Error at index {i}: {e}")

# Add paths to dataframe
if len(audio_paths) < 30:
    print(f"WARNING: Only {len(audio_paths)} audio files generated (expected 30)")
audio_df = audio_df.iloc[:len(audio_paths)].copy()
audio_df['audio_path'] = audio_paths

#  4c. Audio feature extraction
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv
features = []

for i, row in audio_df.iterrows():
    try:
        y, sr = librosa.load(row['audio_path'], sr=None)

        duration = librosa.get_duration(y=y, sr=sr)
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
        zcr = librosa.feature.zero_crossing_rate(y).mean()
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)

        feature_dict = {
            "review": row['review'],
            "label": row['label'],
            "duration": duration,
            "spectral_centroid": centroid,
            "zero_crossing_rate": zcr
        }

        for j in range(13):
            feature_dict[f"mfcc_{j+1}"] = mfcc[j]

        features.append(feature_dict)

    except Exception as e:
        print(f"Feature extraction error at {i}: {e}")

audio_features_df = pd.DataFrame(features)
audio_features_df.to_csv("audio_features.csv", index=False)

print("audio_features.csv saved")

#  4d. Whisper round-trip validation
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

# Function to compute WER
def compute_wer(ref, hyp):
    import re
    ref_words = re.findall(r'\w+', ref.lower())
    hyp_words = re.findall(r'\w+', hyp.lower())

    d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]

    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j

    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1

            d[i][j] = min(
                d[i-1][j] + 1,
                d[i][j-1] + 1,
                d[i-1][j-1] + cost
            )

    if len(ref_words) == 0:
        return 0
    return d[len(ref_words)][len(hyp_words)] / len(ref_words)


validation_results = []

for i, row in audio_df.iterrows():
    try:
        result = _whisper_model.transcribe(row['audio_path'], fp16=False)
        transcription = result['text'].strip()

        wer = compute_wer(row['review'], transcription)
        flag = "FAIL" if wer > 0.25 else "PASS"

        validation_results.append({
            "review": row['review'],
            "label": row['label'],
            "transcription": transcription,
            "wer": wer,
            "quality_flag": flag
        })

    except Exception as e:
        print(f"Error at index {i}: {e}")

audio_validation_df = pd.DataFrame(validation_results)
if len(audio_validation_df) == 0:
    print("WARNING: No validation results generated!")
audio_validation_df.to_csv("audio_validation.csv", index=False)

print("audio_validation.csv saved")
print("PASS rate:", (audio_validation_df['quality_flag'] == "PASS").mean())
print("Average WER:", audio_validation_df['wer'].mean())

30
audio_features.csv saved


100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 179MiB/s]


audio_validation.csv saved
PASS rate: 0.9666666666666667
Average WER: 0.012121212121212121


In [12]:
len(audio_df)
len(audio_validation_df)

30

## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [15]:
!ls

audio_features.csv	 diversity_report.txt	      prompt_template.txt
audio_samples		 evaluator.py		      __pycache__
audio_validation.csv	 final_augmented_dataset.csv  requirements.txt
augmented_classical.csv  gold_standard_100.csv	      sample_data
bilingual_reviews.csv	 llm_generated_300.csv	      text_embedder.pt
class_distribution.png	 llm_generated_flagged.csv    weak_labels_200.csv
consolidated_base.csv	 llm_labels_150.csv


In [14]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv

consolidated_base = pd.read_csv("consolidated_base.csv")
augmented_classical = pd.read_csv("augmented_classical.csv")
llm_generated = pd.read_csv("llm_generated_300.csv")
flagged = pd.read_csv("llm_generated_flagged.csv")
bilingual = pd.read_csv("bilingual_reviews.csv")

# Remove flagged LLM reviews
llm_clean = llm_generated[~llm_generated['review'].isin(flagged['review'])]
# Keep only needed columns
if 'sentiment' in llm_clean.columns:
    llm_clean = llm_clean.rename(columns={'sentiment': 'label'})

llm_clean = llm_clean[['review', 'label']]

# duplicate safety
llm_clean = llm_clean.drop_duplicates(subset=['review'])

# Back translated data
bilingual_clean = bilingual[
    (bilingual['quality_flag'] == 'PASS') &
    (bilingual['sentiment_preserved'] == True)
][['back_translated', 'label']].rename(
    columns={'back_translated': 'review'}
)
# Merge all
final_augmented = pd.concat([
    consolidated_base[['review', 'label']],
    augmented_classical[['review', 'label']],
    llm_clean,
    bilingual_clean
])

# Deduplicate
final_augmented = final_augmented.drop_duplicates(subset=['review']).reset_index(drop=True)

# Save
final_augmented.to_csv("final_augmented_dataset.csv", index=False)

print("final_augmented_dataset.csv saved")
print("Final size:", len(final_augmented))

#  5b. Black-Box Evaluation
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)
baseline_acc = evaluator.run_evaluation(consolidated_base[['review','label']], test_df)
# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)
augmented_acc = evaluator.run_evaluation(final_augmented[['review','label']], test_df)
# Print comparison
print(f"Baseline accuracy:  {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")
print("Final dataset breakdown:")
print(final_augmented['label'].value_counts())

final_augmented_dataset.csv saved
Final size: 633
Initializing Black-Box Embedder...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Embedder loaded successfully.

--- Evaluating: Model ---
Training on 228 samples (excluded 100 test overlaps)...
Accuracy: 76.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.84      0.65        25
     Neutral       0.86      0.68      0.76        47
    Positive       1.00      0.82      0.90        28

    accuracy                           0.76       100
   macro avg       0.80      0.78      0.77       100
weighted avg       0.82      0.76      0.77       100

----------------------------------------

--- Evaluating: Model ---
Training on 533 samples (excluded 100 test overlaps)...
Accuracy: 87.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.96      0.84        25
     Neutral       0.93      0.83      0.88        47
    Positive       0.92      0.86      0.89        28

    accuracy                           0.87       100
   macro avg       0.87      0